## Adding Data in Firestore

# Introduction to Firestore Data Manipulation

Welcome back! Having explored the basics of collections and documents, we will now focus on **Data Manipulation**. You will learn how to create documents, perform efficient bulk uploads, and handle write scenarios safely.

For our examples, we'll use a `students` collection. Each document represents a student with the fields: `student_id`, `name`, `age`, and `major`.

| student_id | name | age | major |
| --- | --- | --- | --- |
| 1 | Alice | 20 | Computer Science |
| 2 | Bob | 22 | Mathematics |

---

## Creating a Document in Firestore

To add a new record, you create a document within a collection. You can either let Firestore generate a unique ID or specify your own (such as using the `student_id`).

The `set()` method is used to create or overwrite a document.

```python
from google.cloud import firestore

db = firestore.Client()

# Add a new student with student_id '3' as the document ID
db.collection('students').document('3').set({
    'student_id': 3,
    'name': 'John',
    'age': 21,
    'major': 'Physics'
})

```

### Key Considerations:

* **Document Size Limits:** Each document can be up to **1 MiB**.
* **Overwriting:** By default, `set()` will overwrite an existing document with the same ID.

---

## Adding Multiple Documents with Batched Writes

When adding several records at once, **Batched Writes** are more efficient. They send multiple operations in a single network request and ensure **atomicity** (all operations succeed or all fail).

```python
from google.cloud import firestore

db = firestore.Client()
batch = db.batch()

# Prepare data
students = [
    {'student_id': 4, 'name': 'Emma', 'age': 23, 'major': 'Biology'},
    {'student_id': 5, 'name': 'Liam', 'age': 22, 'major': 'Chemistry'}
]

for student in students:
    doc_ref = db.collection('students').document(str(student['student_id']))
    batch.set(doc_ref, student)

# Commit the batch to the database
batch.commit()

```

### Limitations:

* **Maximum Operations:** A single batch can contain up to **500** write operations.
* **Atomicity:** If one operation in the batch fails (e.g., due to permissions), the entire batch is rolled back.

---

## Conditional Writes

To prevent accidental overwrites, you can use the `create()` method. It ensures a document is only added if the ID does not already exist.

```python
from google.cloud import firestore
from google.api_core.exceptions import AlreadyExists

db = firestore.Client()
doc_ref = db.collection('students').document('3')

try:
    doc_ref.create({
        'student_id': 3,
        'name': 'John',
        'age': 21,
        'major': 'Physics'
    })
    print("Document created successfully.")
except AlreadyExists:
    print("Document already exists.")

```

---

## Handling Errors When Writing Data

Network issues or exceeding service limits can cause writes to fail. Use `try/except` blocks with `GoogleAPICallError` to handle these gracefully.

```python
from google.cloud import firestore
from google.api_core.exceptions import GoogleAPICallError

db = firestore.Client()
doc_ref = db.collection('students').document('3')

try:
    doc_ref.set({'student_id': 3, 'name': 'John', 'age': 21, 'major': 'Physics'})
    print("Document added successfully.")
except GoogleAPICallError as e:
    print("Failed to add document:", e)

```

---

## Retrieving All Documents in a Collection

To verify your writes, use the `stream()` method to pull every document from a collection.

```python
from google.cloud import firestore

db = firestore.Client()
students_ref = db.collection('students')
docs = students_ref.stream()

for doc in docs:
    print(doc.to_dict())

```

**Example Output:**

> `{'student_id': 1, 'name': 'Alice', 'age': 20, 'major': 'Computer Science'}`
> `{'student_id': 2, 'name': 'Bob', 'age': 22, 'major': 'Mathematics'}`
> ...

---

## Summary and Upcoming Topics

In this lesson, you mastered:

* Adding individual documents with `set()`.
* Using **Batched Writes** for efficiency and atomicity.
* Enforcing data integrity with **Conditional Writes** (`create()`).
* Graceful error handling and verification with `stream()`.

In the next lessons, we will dive into **retrieving specific data**, **updating existing records**, and **deleting documents**. Keep practicing!

What kind of data structure are you planning to build for your first major Firestore project?

## Firestore Student Data Insertion and Collection Creation

Welcome to your first task on data insertion in Firestore! Your task involves running a given script that creates a Firestore collection named Students and populates it with records for three students. The script will add student documents, each including fields like student_id, name, age, and major.

Remember, no coding is required for this task. You'll simply run the given script and observe the changes in the Firestore collection.

Important Note: Running scripts can modify the resources in our GCP simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
from google.cloud import firestore
from google.api_core.exceptions import AlreadyExists

# Create a Firestore client
db = firestore.Client()

# Get reference to the Students collection (collections are created automatically in Firestore)
students_ref = db.collection('Students')

# Put a data item in the collection with set()
students_ref.document('1').set({
    'student_id': 1,
    'name': 'John Doe',
    'age': 22,
    'major': 'Computer Science'
})

# Put multiple data items in the collection with batch write
batch = db.batch()
batch.set(students_ref.document('2'), {
    'student_id': 2,
    'name': 'Jane Doe',
    'age': 21,
    'major': 'Mathematics'
})
batch.set(students_ref.document('3'), {
    'student_id': 3,
    'name': 'Jim Smith',
    'age': 23,
    'major': 'Physics'
})
batch.commit()

# Adding another item with a condition to ensure it doesn't overwrite an existing item
try:
    students_ref.document('1').create({
        'student_id': 1,
        'name': 'Jake Long',
        'age': 24,
        'major': 'Biology'
    })
    print("Item added successfully.")
except AlreadyExists:
    print("Item already exists with the same student_id.")
except Exception as e:
    raise

# List all items in the created collection
docs = students_ref.stream()

for doc in docs:
    print(doc.to_dict())

```

## Updating Student Document in Firestore

Great job! This time, we are refining our interaction skills with Google Cloud Firestore. Before running the provided Python script, you are tasked with modifying it. Specifically, you need to update the Students collection that we've been working with by modifying the attributes of a particular student document. Here's what you need to do:

    Change the name attribute from John Doe to John Smith for the student with student_id of 1.
    Add a new attribute email with the value john.smith@example.com for the same student document.

Important Note: Running scripts can modify the resources in our GCP simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
from google.cloud import firestore

# Create a Firestore client
db = firestore.Client()

# Reference to the Students collection
students_collection = db.collection('Students')

# TODO: Modify the set() method below to change the name from 'John Doe' to 'John Smith' and add a new attribute 'email' with the value 'john.smith@example.com'

# Create a document in the Students collection
students_collection.document('1').set({
    'student_id': 1,
    'name': 'John Doe',
    'age': 22,
    'major': 'Computer Science'
    # Add new attribute 'email' here
})

# List all documents in the Students collection
docs = students_collection.stream()

for doc in docs:
    print(f'{doc.id} => {doc.to_dict()}')

```

To update the document as requested, you will modify the dictionary passed to the `set()` method. In Firestore, when you use `set()` on an existing document ID, it replaces the current data with the new data you provide.

Here is the updated script to change the name and add the email address:

```python
from google.cloud import firestore

# Create a Firestore client
db = firestore.Client()

# Reference to the Students collection
students_collection = db.collection('Students')

# --- TODO FIXED: Update the document attributes ---

# Create or overwrite the document with ID '1'
students_collection.document('1').set({
    'student_id': 1,
    'name': 'John Smith',               # Updated from 'John Doe'
    'email': 'john.smith@example.com',  # New attribute added
    'age': 22,
    'major': 'Computer Science'
})

# --------------------------------------------------

# List all documents in the Students collection to verify the update
docs = students_collection.stream()

print("Current Students in collection:")
for doc in docs:
    print(f'{doc.id} => {doc.to_dict()}')

```

### Pro-Tip: `set()` vs `update()`

* **`set()`**: Replaces the **entire document**. If you forgot to include the `age` or `major` in the code above, those fields would actually be deleted from the database.
* **`update()`**: If the document already exists and you only want to change specific fields (without sending the whole dictionary again), you can use `students_collection.document('1').update({'name': 'John Smith'})`. This is often safer for large documents where you only need to change one small piece of data.

Since we are mastering the basics here, using `set()` ensures the document is exactly how you defined it in your code!

## Adding a Student Record to Firestore Collection

In the following task, you are now asked to take a step further and interact with the Students Firestore collection you created earlier. Your goal for this task is to add one more student record to the collection. Begin by reviewing the provided script, and when you're ready, fill in the TODO line of code with a new student record, adding another student named Emily Johnson with the details student_id: 2, name: 'Emily Johnson', age: 22, and major: 'Data Science' to the Students collection.

Please ensure your script runs without errors and successfully populates the database with the new record.

Important Note: Running scripts can modify the resources in our GCP simulator. To revert to the initial state, you can use the reset button located in the top right corner. Keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
from google.cloud import firestore

# Create a Firestore client
db = firestore.Client()

# Reference to the Students collection
students_collection = db.collection('Students')

# Create a document in the Students collection
students_collection.document('1').set({
    'student_id': 1,
    'name': 'John Smith',
    'age': 22,
    'major': 'Computer Science'
})

# TODO: Add another student 'Emily Johnson' with details student_id: 2, age: 22 and major: 'Data Science'

# List all documents in the Students collection
docs = students_collection.stream()

for doc in docs:
    print(f'{doc.id} => {doc.to_dict()}')

```

To add Emily Johnson to your database, you will use the same `.document().set()` pattern. By specifying `'2'` as the document ID, you keep the collection organized and easy to query by ID later.

Here is the completed script:

```python
from google.cloud import firestore

# Create a Firestore client
db = firestore.Client()

# Reference to the Students collection
students_collection = db.collection('Students')

# Create the first document
students_collection.document('1').set({
    'student_id': 1,
    'name': 'John Smith',
    'age': 22,
    'major': 'Computer Science'
})

# --- TODO FIXED: Add Emily Johnson ---
students_collection.document('2').set({
    'student_id': 2,
    'name': 'Emily Johnson',
    'age': 22,
    'major': 'Data Science'
})

# List all documents in the Students collection to verify
docs = students_collection.stream()

print("Current Students in collection:")
for doc in docs:
    print(f'{doc.id} => {doc.to_dict()}')

```

### Key Concept: The "Write" Operation

When you call `.set()`, the Firestore client sends a request to the server. If the collection `Students` didn't exist yet, Firestore creates it automatically on the fly. This "schema-less" nature is what makes NoSQL so powerful for rapid iteration—you can add a third student with completely different fields (like a `minor` or `gpa`) without having to modify the database structure first.

## Firestore Batch Operations for Student Documents

Great progress! Now that you're familiar with Firestore basics, it's time to dive into a practical coding exercise. In this task, we have pre-provided a script for you that initializes a Firestore client and sets up a reference to a collection named Students. Your primary task is to utilize batch operations to add three student documents simultaneously. Each document should contain the fields: student_id, name, age, and major. After successfully committing the batch operation, print a confirmation message and then retrieve and display all student documents from the collection to verify the data was added correctly.

Important Note: Running scripts can modify the resources in our GCP simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
from google.cloud import firestore

# Create a Firestore client
db = firestore.Client()

# Reference to the Students collection
students_collection = db.collection('Students')

# TODO: Use batch operations to add three student documents to the collection.
# TODO: Print a confirmation message after the batch is committed.
# TODO: Retrieve and display all documents in the collection to verify the operation.
```

Batch operations are a powerful feature in Firestore that allow you to group multiple write operations into a single atomic transaction. This means that either all the documents are added successfully, or none of them are—ensuring your database stays consistent even if a network error occurs mid-process.

Here is the completed script using the `write_batch` object:

```python
from google.cloud import firestore

# Create a Firestore client
db = firestore.Client()

# Reference to the Students collection
students_collection = db.collection('Students')

# 1. Initialize the batch
batch = db.batch()

# 2. Define the student data
new_students = [
    {'id': '3', 'data': {'student_id': 3, 'name': 'Michael Chen', 'age': 20, 'major': 'Physics'}},
    {'id': '4', 'data': {'student_id': 4, 'name': 'Sarah Miller', 'age': 21, 'major': 'Architecture'}},
    {'id': '5', 'data': {'student_id': 5, 'name': 'David Wilson', 'age': 23, 'major': 'History'}}
]

# 3. Add operations to the batch
for student in new_students:
    doc_ref = students_collection.document(student['id'])
    batch.set(doc_ref, student['data'])

# 4. Commit the batch
batch.commit()

# Print confirmation
print("Batch operation committed successfully: 3 students added.")

# 5. Retrieve and display all documents to verify
print("\n--- Verifying Students Collection ---")
docs = students_collection.stream()

for doc in docs:
    print(f'Document ID: {doc.id} => Data: {doc.to_dict()}')

```

### Key Batch Mechanics

* **Efficiency**: Instead of making three separate trips to the server, you only make one. This significantly reduces latency when dealing with multiple records.
* **Atomicity**: If you had 500 operations in a batch and the 499th one failed, Firestore would roll back the previous 498, leaving your database exactly as it was before the script started.
* **The 500 Limit**: Keep in mind that a single Firestore batch is limited to **500 operations**. If you need to upload thousands of documents, you would need to split them into multiple batches.

## Google Cloud Firestore Students Collection Management

This is the final task for this lesson! You've been doing great so far, and now it's time to put everything together. In this exercise, you are tasked with interacting with an existing Google Cloud Firestore collection named Students, which already has student_id as the document ID. Populate this collection with a few records, ensuring each record includes student_id, name, age, and major. Utilize conditional expressions to prevent overwriting existing records. Lastly, query and print all items from the Students collection.

Ready to take on the challenge? Remember all the skills and knowledge you've acquired so far, and good luck!

Important Note: Running scripts can modify the resources in our Firestore database. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
from google.cloud import firestore

# Create a Firestore client
db = firestore.Client()

# Reference to the Students collection
students_collection = db.collection('Students')

# Use batch operations to add three student documents simultaneously
batch = db.batch()
batch.set(students_collection.document('1'), {'student_id': 1, 'name': 'John Doe', 'age': 22, 'major': 'Computer Science'})
batch.set(students_collection.document('2'), {'student_id': 2, 'name': 'Jane Doe', 'age': 21, 'major': 'Mathematics'})
batch.set(students_collection.document('3'), {'student_id': 3, 'name': 'Jim Smith', 'age': 23, 'major': 'Physics'})
batch.commit()

# TO DO: Add a student item to the collection
# TO DO: Use a batch operation to add multiple student items to the collection
# TO DO: Try to add another item with the same document ID as an existing item using a condition expression to avoid overwriting
# TO DO: List and print all items in the created collection

```